In [84]:
import pandas as pd
import gc

## Leitura dos Dados e Remoção do Cabeçalho

A planilha do IBGE possui as informações estruturadas a partir da linha 9 (índice 8). Lemos o arquivo pulando as 8 primeiras linhas de cabeçalhos descritivos.

In [85]:
# Lemos o arquivo pulando as 8 primeiras linhas de cabeçalho do Excel
df_censo_raca = pd.read_excel('censo_2022/dados_raciais_2022_MU.xlsx', skiprows=8, header=None)
df_censo_municipio = pd.read_csv('censo_2022/censo_2022_dados_municipais.csv', sep=",", encoding="latin-1")

print(f"Tamanho inicial do dataset: {df_censo.shape}")
display(df_censo_raca.head())

Tamanho inicial do dataset: (5570, 15)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Absoluto,Percentual
1,Alta Floresta D'Oeste,1100015.0,21 494,7 590,1 297,46,11 845,715,1,35.313823,6.034523,0.214023,55.110966,3.326664,749,3.484693
2,Ariquemes,1100023.0,96 833,31 263,9 619,317,55 534,97,3,32.286481,9.933905,0.327378,57.352060,0.100176,98,0.101205
3,Cabixi,1100031.0,5 351,2 136,345,12,2 856,2,0,39.917772,6.447393,0.224257,53.373201,0.037376,10,0.186881
4,Cacoal,1100049.0,86 887,32 304,6 704,365,45 836,1 675,3,37.180609,7.716035,0.420100,52.755398,1.927858,1688,1.942753


In [86]:
df_censo_municipio.head(10)

,id_municipio,sigla_uf,domicilios,populacao,area,taxa_alfabetizacao,idade_mediana,razao_sexo,indice_envelhecimento,populacao_indigena,populacao_indigena_terra_indigena,populacao_quilombola,populacao_quilombola_territorio_quilombola
0,1400704,RR,2339,13695,8114,0.87529,15.0,105.76,5.40,13264,12213,0,0
1,1200435,AC,1344,6717,6156,0.77397,17.0,103.73,7.68,4297,2639,0,0
2,1400407,RR,2719,13893,6960,0.91823,17.0,106.53,8.23,12397,10081,0,0
3,1200328,AC,2003,9201,5357,0.81876,18.0,107.10,6.55,4109,3273,0,0
4,1303908,AM,6010,32931,19659,0.85195,19.0,105.88,11.12,26603,12081,0,0
5,1302306,AM,4845,24981,69457,0.84097,19.0,106.14,9.52,8148,3798,0,0
6,1504505,PA,5715,27714,6774,0.80117,19.0,109.43,9.25,11,0,0,0
7,1501105,PA,5748,31715,4397,0.82553,19.0,103.93,8.67,1,0,869,463
8,1400027,RR,2996,13715,28473,0.68174,19.0,110.00,11.29,9651,8966,0,0
9,1200393,AC,2611,10735,6446,0.81098,20.0,110.37,9.98,868,654,0,0


## Seleção e Renomeação das Colunas

Com base na inspeção da estrutura do arquivo de raça do IBGE, identificamos as colunas de interesse:
- Índice 0: Nome do Município
- Índice 1: Código do Município
- Índice 2: População Total
- Índice 3: População Branca
- Índice 4: População Preta 
- Índice 5: População Amarela
- Índice 6: População Parda
- Índice 7: População Indígena

Selecionamos apenas essas colunas e as renomeamos para o padrão adotado no projeto.

Quanto aos dados municipais, escolhemos:
- Código do Município
- sigla_uf
- n° domicilios
- população
- área
- taxa de alfabetização
- idade mediana
- razão sexo
- índice de envelhecimento

In [87]:
# Selecionamos as colunas pelos índices numéricos
colunas_interesse = [0, 1, 2, 3, 4, 5, 6, 7]
df_censo_raca = df_censo_raca[colunas_interesse].copy()

# Dicionário de renomeação para padronizar as variáveis
colunas_map = {
    0: 'city_name',
    1: 'city_code',
    2: 'population_total',
    3: 'population_white',
    4: 'population_black',
    5: 'population_yellow',
    6: 'population_brown',
    7: 'population_indigenous'
}

# Aplicamos o novo nome às colunas
df_censo_raca = df_censo_raca.rename(columns=colunas_map)

display(df_censo_raca.head())

,city_name,city_code,population_total,population_white,population_black,population_yellow,population_brown,population_indigenous
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Alta Floresta D'Oeste,1100015.0,21 494,7 590,1 297,46,11 845,715
2,Ariquemes,1100023.0,96 833,31 263,9 619,317,55 534,97
3,Cabixi,1100031.0,5 351,2 136,345,12,2 856,2
4,Cacoal,1100049.0,86 887,32 304,6 704,365,45 836,1 675


In [88]:
# Remove as colunas que iniciam com dados de população antes do merge
cols_populacao_sub = [col for col in df_censo_municipio.columns if col.startswith('populacao')]
df_censo_municipio = df_censo_municipio.drop(columns=cols_populacao_sub, errors='ignore')

# Dicionário de tradução e padronização para o inglês
colunas_renomear = {
    'id_municipio': 'city_code',
    'sigla_uf': 'state_abbreviation',
    'domicilios': 'households',
    'area': 'area',
    'taxa_alfabetizacao': 'literacy_rate',
    'idade_mediana': 'median_age',
    'razao_sexo': 'sex_ratio',
    'indice_envelhecimento': 'aging_index'
}


# Filtra e renomeia mantendo apenas as colunas válidas presentes no seu DataFrame
colunas_manter = [c for c in colunas_renomear.keys() if c in df_censo_municipio.columns]
df_censo_municipio = df_censo_municipio[colunas_manter].rename(columns=colunas_renomear)

# Validação final
print("df_censo_municipio tratado com sucesso!")
print(f"Total de registros: {df_censo_municipio.shape[0]}")
print(f"Colunas atuais: {df_censo_municipio.columns.tolist()}")

df_censo_municipio tratado com sucesso!
Total de registros: 5570
Colunas atuais: ['city_code', 'state_abbreviation', 'households', 'area', 'literacy_rate', 'median_age', 'sex_ratio', 'aging_index']


## Limpeza de Rodapés e Conversão de Tipos

Os arquivos do IBGE costumam trazer informações textuais (fontes, notas técnicas) nas últimas linhas, o que converte as colunas numéricas para o tipo `object` (string). Vamos limpar esses rodapés forçando a conversão da coluna `city_code` para numérico. Linhas de rodapé se tornarão nulas (`NaN`) e poderão ser removidas.

In [89]:
# Forçamos a conversão da coluna 'city_code' para numérico.
# Textos de rodapé do IBGE retornarão NaN.
df_censo_raca['city_code'] = pd.to_numeric(df_censo_raca['city_code'], errors='coerce')

# Removemos apenas as linhas onde o 'city_code' é nulo (rodapés reais)
df_censo_raca = df_censo_raca.dropna(subset=['city_code']).copy()

# Colunas populacionais que serão tratadas
colunas_inteiras = [
    'city_code', 'population_total', 'population_white', 
    'population_black', 'population_yellow', 'population_brown', 'population_indigenous'
]

# Ajuste de tipo para o código do município (7 dígitos)
if 'city_code' in df_censo_municipio.columns:
    df_censo_municipio['city_code'] = df_censo_municipio['city_code'].astype('int32')


for col in colunas_inteiras:
    # 1. Converte para string, remove espaços e troca hifens/traços do IBGE por zero
    df_censo_raca[col] = df_censo_raca[col].astype(str).str.replace(' ', '').str.replace('-', '0')
    
    # 2. Converte para numérico (qualquer texto estranho remanescente vira NaN)
    df_censo_raca[col] = pd.to_numeric(df_censo_raca[col], errors='coerce')
    
    # 3. Preenche os NaNs resultantes (ausência de dados) com 0
    df_censo_raca[col] = df_censo_raca[col].fillna(0)
    
    # 4. Converte para inteiro
    df_censo_raca[col] = df_censo_raca[col].astype(int)

print(f"Total de municípios após a limpeza: {len(df_censo_raca)}")

Total de municípios após a limpeza: 5570


In [90]:
df_censo_raca

,city_name,city_code,population_total,population_white,population_black,population_yellow,population_brown,population_indigenous
1,Alta Floresta D'Oeste,1100015,21494,7590,1297,46,11845,715
2,Ariquemes,1100023,96833,31263,9619,317,55534,97
3,Cabixi,1100031,5351,2136,345,12,2856,2
4,Cacoal,1100049,86887,32304,6704,365,45836,1675
5,Cerejeiras,1100056,15890,6213,1025,59,8580,13
...,...,...,...,...,...,...,...,...
5566,Vianópolis,5222005,14956,6328,954,14,7645,15
5567,Vicentinópolis,5222054,8768,3349,715,10,4683,11
5568,Vila Boa,5222203,4215,899,861,3,2444,8
5569,Vila Propício,5222302,5815,1407,533,24,3844,5


In [91]:
# Executa o merge combinando as duas bases por 'city_code' e armazena em df_censo
df_censo = pd.merge(
    df_censo_municipio,
    df_censo_raca,
    on='city_code',
    how='left'
)

print("Merge realizado e arquivo salvo com sucesso!")
print(f"Dimensões finais: {df_censo.shape}")
print(f"Colunas finais: {df_censo.columns.tolist()}")

Merge realizado e arquivo salvo com sucesso!
Dimensões finais: (5570, 15)
Colunas finais: ['city_code', 'state_abbreviation', 'households', 'area', 'literacy_rate', 'median_age', 'sex_ratio', 'aging_index', 'city_name', 'population_total', 'population_white', 'population_black', 'population_yellow', 'population_brown', 'population_indigenous']


## Exportação do Dataset Preparado

Com o arquivo limpo, estruturado e tipado, nós o exportamos no formato CSV para que seja integrado rapidamente no notebook principal de EDA.

In [92]:
# Exportação para CSV
df_censo.to_csv('censo_2022/censo_2022.csv', index=False, sep=';', encoding='utf-8')